#**In this lesson we'll learn two Object Tracking Algorithms:**
1. How to use the Mean Shift Algorithm in OpenCV
2. Use CAMSHIFT in OpenCV

In [15]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from google.colab.patches import cv2_imshow
from IPython.display import HTML
from base64 import b64encode

In [16]:
!wget https://github.com/makelove/OpenCV-Python-Tutorial/raw/master/data/slow.flv

--2025-02-27 14:35:38--  https://github.com/makelove/OpenCV-Python-Tutorial/raw/master/data/slow.flv
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/makelove/OpenCV-Python-Tutorial/master/data/slow.flv [following]
--2025-02-27 14:35:38--  https://raw.githubusercontent.com/makelove/OpenCV-Python-Tutorial/master/data/slow.flv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1238488 (1.2M) [video/flv]
Saving to: ‘slow.flv.2’

slow.flv.2          100%[===================>]   1.18M  --.-KB/s    in 0.01s   

2025-02-27 14:35:38 (123 MB/s) - ‘slow.flv.2’ saved [1238488/1238488]



# **Meanshif Object Tracking**

![](https://opencv-python-tutroals.readthedocs.io/en/latest/_images/meanshift_basics.jpg)

The intuition behind the meanshift is simple. Consider you have a set of points. (It can be a pixel distribution like histogram backprojection). You are given a small window ( may be a circle) and you have to move that window to the area of maximum pixel density (or maximum number of points). It is illustrated in the simple image given below:

![](https://opencv-python-tutroals.readthedocs.io/en/latest/_images/meanshift_face.gif)

Mean shift is a hill climbing algorithm which involves shifting this kernel iteratively to a higher density region until convergence. Every shift is defined by a mean shift vector. The mean shift vector always points toward the direction of the maximum increase in the density.
![](https://upload.wikimedia.org/wikipedia/commons/b/bd/Meanshiftred.gif)

Read Paper Here - https://ieeexplore.ieee.org/document/732882

Animation Source - https://fr.wikipedia.org/wiki/Camshift

In [17]:
import cv2
import numpy as np

def setup_tracker(video_path, roi_size_factor=0.2, min_window_size=20, max_window_factor=0.5):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError("Could not open video file")

    ret, frame = cap.read()
    if not ret:
        raise ValueError("Could not read first frame")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    output_path = 'tracking_output.avi'
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    roi_width = max(int(width * roi_size_factor), min_window_size)
    roi_height = max(int(height * roi_size_factor), min_window_size)
    x = max(0, (width - roi_width) // 2)
    y = max(0, (height - roi_height) // 2)
    track_window = (x, y, roi_width, roi_height)

    roi = frame[y:y+roi_height, x:x+roi_width]
    if roi.size == 0:
        raise ValueError("Invalid ROI dimensions")

    hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv_roi, np.array([0, 60, 32]), np.array([180, 255, 255]))
    roi_hist = cv2.calcHist([hsv_roi], [0], mask, [180], [0, 180])
    cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

    term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 15, 1)  # More iterations for better tracking
    max_lost_frames = int(fps * 3)  # Increased tolerance for lost frames

    return cap, out, track_window, roi_hist, term_crit, {
        'width': width,
        'height': height,
        'min_size': min_window_size,
        'max_size': max(width, height) * max_window_factor,
        'max_lost': max_lost_frames
    }

def track_video_to_avi(video_path):
    try:
        cap, out, track_window, roi_hist, term_crit, params = setup_tracker(video_path)
        lost_counter = 0
        reset_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                print("End of video - AVI saved")
                break

            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
            hsv = cv2.GaussianBlur(hsv, (5, 5), 0)  # Smooth image for better tracking
            dst = cv2.calcBackProject([hsv], [0], roi_hist, [0, 180], 1)

            _, track_window = cv2.meanShift(dst, track_window, term_crit)
            x, y, w, h = track_window

            # Check if tracking is lost or unstable
            if w < params['min_size'] or h < params['min_size'] or w > params['max_size'] or h > params['max_size']:
                lost_counter += 1
            else:
                lost_counter = max(0, lost_counter - 1)

            x = max(0, min(x, params['width'] - w))
            y = max(0, min(y, params['height'] - h))
            track_window = (x, y, w, h)

            output_frame = frame.copy()
            cv2.rectangle(output_frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

            if lost_counter > params['max_lost'] // 2:
                status = f"Lost tracking ({lost_counter}) - Resetting ROI"
                color = (0, 0, 255)
                reset_count += 1

                # Reset to the center if tracking is lost for too long
                roi_width = max(int(params['width'] * 0.2), params['min_size'])
                roi_height = max(int(params['height'] * 0.2), params['min_size'])
                x, y = (params['width'] - roi_width) // 2, (params['height'] - roi_height) // 2
                track_window = (x, y, roi_width, roi_height)
                lost_counter = 0  # Reset counter

            elif lost_counter > 0:
                status = f"Unstable tracking ({lost_counter})"
                color = (0, 165, 255)
            else:
                status = "Tracking (MeanShift)"
                color = (0, 255, 0)

            # Display tracking status
            cv2.putText(output_frame, status, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            cv2.putText(output_frame, f"Resets: {reset_count}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            out.write(output_frame)

        cap.release()
        out.release()
        return "tracking_output.avi"

    except Exception as e:
        print(f"Error: {e}")
        if 'cap' in locals():
            cap.release()
        if 'out' in locals():
            out.release()
        return None

video_path = '/content/slow.flv'  # Replace with your video path
avi_file = track_video_to_avi(video_path)
print(f"Step 1 complete: AVI saved as {avi_file}")


End of video - AVI saved
Step 1 complete: AVI saved as tracking_output.avi


In [18]:
!ffmpeg -i /content/tracking_output.avi car_tracking_mean_shift.mp4 -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [19]:

mp4 = open('/content/car_tracking_mean_shift.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML("""
<video controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

## **Camshift in OpenCV**
It is almost same as meanshift, but it returns a rotated rectangle (that is our result) and box parameters (used to be passed as search window in next iteration).

![](https://upload.wikimedia.org/wikipedia/commons/8/86/CamshiftStillImage.gif)

Read Paper Here - https://ieeexplore.ieee.org/document/732882

Animation Source - https://fr.wikipedia.org/wiki/Camshift

In [20]:
cap = cv2.VideoCapture('/content/slow.flv')

# take first frame of the video
ret,frame = cap.read()

# Get the height and width of the frame (required to be an interger)
width = int(cap.get(3))
height = int(cap.get(4))

# Define the codec and create VideoWriter object. The output is stored in '*.avi' file.
out = cv2.VideoWriter('camshift_car_tracking_cam_shift.avi', cv2.VideoWriter_fourcc('M','J','P','G'), 30, (width, height))

# setup initial location of window
r,h,c,w = 250,90,400,125  # simply hardcoded the values
track_window = (c,r,w,h)

# set up the ROI for tracking
roi = frame[r:r+h, c:c+w]
hsv_roi =  cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(hsv_roi, np.array((0., 60.,32.)), np.array((180.,255.,255.)))
roi_hist = cv2.calcHist([hsv_roi],[0],mask,[180],[0,180])
cv2.normalize(roi_hist,roi_hist,0,255,cv2.NORM_MINMAX)

# Setup the termination criteria, either 10 iteration or move by atleast 1 pt
term_crit = ( cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1 )

while(1):
    ret ,frame = cap.read()

    if ret == True:
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        dst = cv2.calcBackProject([hsv],[0],roi_hist,[0,180],1)

        # apply meanshift to get the new location
        ret, track_window = cv2.CamShift(dst, track_window, term_crit)

        # Draw it on image
        pts = cv2.boxPoints(ret)
        pts = np.int0(pts)
        img2 = cv2.polylines(frame,[pts],True, 255,2)
        out.write(img2)
        #imshow('img2',img2)

    else:
        break

cap.release()
out.release()

<ipython-input-20-45c49e94f7c6>:39: DeprecationWarning: `np.int0` is a deprecated alias for `np.intp`.  (Deprecated NumPy 1.24)
  pts = np.int0(pts)


In [21]:
!ffmpeg -i /content/camshift_car_tracking_cam_shift.avi camshift_car_tracking_cam_shift.mp4 -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [22]:

mp4 = open('/content/camshift_car_tracking_cam_shift.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

In [23]:
HTML("""
<video controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)